In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print("Project root:", project_root)

In [ ]:
import geopandas as gpd
import numpy as np
import rasterio
from rasterstats import zonal_stats

In [ ]:
from pysheds.grid import Grid

print("pysheds import successful")

In [ ]:
grid = Grid.from_raster(dem_path, data_name="dem")

print("pysheds grid created successfully")

In [ ]:
fdir = grid.flowdir(
    grid.dem,
    routing="d8"
)

print("Flow direction computed successfully")

In [ ]:
import inspect

print(inspect.signature(grid.flowdir))
print(inspect.getdoc(grid.flowdir))

In [ ]:
dem_raster = grid.read_raster(dem_path)

print("DEM loaded into pysheds successfully")
print(type(dem_raster))

In [ ]:
fdir = grid.flowdir(
    dem_raster,
    routing="d8"
)

print("Flow direction computed successfully")

In [ ]:
fac = grid.accumulation(
    fdir,
    routing="d8"
)

print("Flow accumulation computed successfully")
print("Flow accumulation shape:", fac.shape)

In [ ]:
slope_radians = np.radians(slope)

safe_slope = np.where(
    slope_radians > 0,
    slope_radians,
    np.nan
)

twi = np.log(
    fac / np.tan(safe_slope)
)

twi[~np.isfinite(twi)] = np.nan

print("TWI computed successfully")
print("TWI shape:", twi.shape)
print("Minimum TWI:", np.nanmin(twi))
print("Maximum TWI:", np.nanmax(twi))
print("Mean TWI:", np.nanmean(twi))

In [ ]:
twi_temp_path = tempfile.NamedTemporaryFile(
    suffix=".tif",
    delete=False
).name

with rasterio.open(dem_path) as src:
    twi_profile = src.profile.copy()

twi_profile.update(
    dtype="float32",
    count=1,
    nodata=np.nan
)

with rasterio.open(
    twi_temp_path,
    "w",
    **twi_profile
) as dst:
    dst.write(twi.astype("float32"), 1)

print("Temporary TWI raster:", twi_temp_path)

In [ ]:
twi_stats = zonal_stats(
    grid_gdf,
    twi_temp_path,
    stats=["mean", "max"],
    nodata=np.nan
)

grid_gdf["twi_mean"] = [
    stat["mean"] for stat in twi_stats
]

grid_gdf["twi_max"] = [
    stat["max"] for stat in twi_stats
]

print(
    grid_gdf[
        [
            "grid_id",
            "twi_mean",
            "twi_max"
        ]
    ].head()
)

In [ ]:
print("Valid TWI mean:", grid_gdf["twi_mean"].notna().sum())
print("Valid TWI max:", grid_gdf["twi_max"].notna().sum())

print("\nExample valid TWI rows:")
print(
    grid_gdf[
        grid_gdf["twi_mean"].notna()
    ][[
        "grid_id",
        "twi_mean",
        "twi_max"
    ]].head()
)

In [ ]:
terrain_features = grid_gdf[
    [
        "grid_id",
        "elevation_mean",
        "elevation_max",
        "slope_mean",
        "slope_max",
        "twi_mean",
        "twi_max",
    ]
].copy()

print("Rows:", len(terrain_features))
print("Columns:", list(terrain_features.columns))
print(terrain_features.head())

In [ ]:
output_path = "../data/processed/terrain_features.parquet"

terrain_features.to_parquet(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows:", len(terrain_features))
print("Columns:", list(terrain_features.columns))

In [ ]:
valid_terrain = grid_gdf[
    grid_gdf["elevation_mean"].notna()
    & grid_gdf["elevation_max"].notna()
    & grid_gdf["slope_mean"].notna()
    & grid_gdf["slope_max"].notna()
]

print("Total grid cells:", len(grid_gdf))
print("Valid terrain cells:", len(valid_terrain))
print("Cells with missing terrain:", len(grid_gdf) - len(valid_terrain))

In [ ]:
print("Elevation:")
print("Min:", grid_gdf["elevation_mean"].min(), "m")
print("Max:", grid_gdf["elevation_mean"].max(), "m")
print("Mean:", grid_gdf["elevation_mean"].mean(), "m")

In [ ]:
print("Slope:")
print("Min:", grid_gdf["slope_mean"].min(), "degrees")
print("Max:", grid_gdf["slope_mean"].max(), "degrees")
print("Mean:", grid_gdf["slope_mean"].mean(), "degrees")

In [ ]:
twi_valid = (
    grid_gdf["twi_mean"].notna()
    & grid_gdf["twi_max"].notna()
)

print("Valid TWI cells:", twi_valid.sum())
print("TWI worked:", twi_valid.sum() > 0)

In [ ]:
np.in1d = np.isin

print("NumPy compatibility fix applied")

In [ ]:
grid_path = "../data/processed/kamrup_metro_grid_1km.parquet"

grid_gdf = gpd.read_parquet(grid_path)

print("Rows:", len(grid_gdf))
print("Columns:", list(grid_gdf.columns))
print("CRS:", grid_gdf.crs)

In [ ]:
dem_path = "../data/raw/kamrup_metro_dem.tif"

with rasterio.open(dem_path) as src:
    dem_crs = src.crs
    dem_transform = src.transform
    dem_shape = src.shape
    nodata = src.nodata

print("DEM CRS:", dem_crs)
print("DEM shape:", dem_shape)
print("DEM NoData:", nodata)

In [ ]:
with rasterio.open(dem_path) as src:
    dem = src.read(1)
    nodata = src.nodata

dem = dem.astype(float)
dem[dem == nodata] = np.nan

print("DEM shape:", dem.shape)
print("NoData value:", nodata)

In [ ]:
elevation_stats = zonal_stats(
    grid_gdf,
    dem_path,
    stats=["mean", "max"],
    nodata=nodata
)

grid_gdf["elevation_mean"] = [
    stat["mean"] for stat in elevation_stats
]

grid_gdf["elevation_max"] = [
    stat["max"] for stat in elevation_stats
]

print(grid_gdf[[
    "grid_id",
    "elevation_mean",
    "elevation_max"
]].head())

In [ ]:
with rasterio.open(dem_path) as src:
    print("DEM bounds:", src.bounds)

print("Grid bounds:", grid_gdf.total_bounds)

In [ ]:
from shapely.geometry import box

with rasterio.open(dem_path) as src:
    dem_bbox = box(*src.bounds)

overlap_mask = grid_gdf.geometry.intersects(dem_bbox)

print("Total grid cells:", len(grid_gdf))
print("Cells overlapping DEM:", overlap_mask.sum())
print("Cells not overlapping DEM:", (~overlap_mask).sum())

In [ ]:
nan_elevation = grid_gdf[
    grid_gdf["elevation_mean"].isna()
][["grid_id", "geometry"]]

print("Cells with NaN elevation_mean:", len(nan_elevation))
print(nan_elevation[["grid_id"]].head(20))

In [ ]:
from app.terrain_utils import compute_slope

with rasterio.open(dem_path) as src:
    bounds = src.bounds
    pixel_width_deg, pixel_height_deg = src.res

center_lat = (bounds.top + bounds.bottom) / 2

meters_per_degree_lat = 111320
meters_per_degree_lon = 111320 * np.cos(np.radians(center_lat))

cellsize_y = pixel_height_deg * meters_per_degree_lat
cellsize_x = pixel_width_deg * meters_per_degree_lon

slope = compute_slope(
    dem,
    (cellsize_y, cellsize_x)
)

print("Slope shape:", slope.shape)
print("Minimum slope:", np.nanmin(slope), "degrees")
print("Maximum slope:", np.nanmax(slope), "degrees")
print("Mean slope:", np.nanmean(slope), "degrees")

In [ ]:
import tempfile

slope_temp_path = tempfile.NamedTemporaryFile(
    suffix=".tif",
    delete=False
).name

with rasterio.open(dem_path) as src:
    slope_profile = src.profile.copy()

slope_profile.update(
    dtype="float32",
    count=1,
    nodata=np.nan
)

with rasterio.open(
    slope_temp_path,
    "w",
    **slope_profile
) as dst:
    dst.write(slope.astype("float32"), 1)

print("Temporary slope raster:", slope_temp_path)

In [ ]:
slope_stats = zonal_stats(
    grid_gdf,
    slope_temp_path,
    stats=["mean", "max"],
    nodata=np.nan
)

grid_gdf["slope_mean"] = [
    stat["mean"] for stat in slope_stats
]

grid_gdf["slope_max"] = [
    stat["max"] for stat in slope_stats
]

print(grid_gdf[[
    "grid_id",
    "slope_mean",
    "slope_max"
]].head())

In [ ]:
print("Valid elevation_mean:", grid_gdf["elevation_mean"].notna().sum())
print("Valid elevation_max:", grid_gdf["elevation_max"].notna().sum())
print("Valid slope_mean:", grid_gdf["slope_mean"].notna().sum())
print("Valid slope_max:", grid_gdf["slope_max"].notna().sum())

print("\nExample valid terrain rows:")
print(
    grid_gdf[
        grid_gdf["elevation_mean"].notna()
    ][[
        "grid_id",
        "elevation_mean",
        "elevation_max",
        "slope_mean",
        "slope_max"
    ]].head()
)

In [ ]:
terrain_features = grid_gdf[
    [
        "grid_id",
        "elevation_mean",
        "elevation_max",
        "slope_mean",
        "slope_max",
    ]
].copy()

print("Rows:", len(terrain_features))
print("Columns:", list(terrain_features.columns))
print(terrain_features.head())

In [ ]:
output_path = "../data/processed/terrain_features.parquet"

terrain_features.to_parquet(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows:", len(terrain_features))